In [1]:
from functools import partial
import os
import math

os.environ["LIBTPU_INIT_ARGS"] = " ".join([
  # "--xla_tpu_enable_sc_log_recorder=true",
  # "--xla_tpu_enable_tile_log_recorder=true",
  "--xla_tpu_use_tc_device_shape_on_sc=true"
])


import jax
import jax.numpy as jnp
from jax import random
import numpy as np

import jax.experimental.pallas as pl
import jax.experimental.pallas.tpu as pltpu
import jax.experimental.pallas.tpu_sc as plsc
from jax.experimental.compute_on import compute_on

# compute_on attempt

In [10]:
@compute_on("tpu_sparsecore")
@jax.jit
def gather_3d_to_2d(x, indices):
  return x.reshape((x.shape[0], -1))
  chunk = x.shape[1] // 4
  return jnp.concat([x[:, i * chunk:(i + 1) * chunk, :][indices, ...] for i in range(4)], axis=-2)
  return x[indices, ...]
  # return x.reshape((x.shape[0], -1))[indices, ...]
  # return x.reshape((x.shape[0], x.shape[1] // 128, 128))


@jax.jit
def combined(x, indices):
  x_ = x[..., :2048]
  return (gather_3d_to_2d(x, indices), x_.mT @ x_)

In [7]:
LANES = 128
NUM_SUBCORES = 16
NUM_CORES = 2

# pad to multiples of (8 * NUM_SUBCORES)
# shape = (262144, 7168 // 2)
shape = (262144, 7168)
# shape = (262144, 7168)
# dtype = jnp.bfloat16
dtype = jnp.float32
x = jnp.arange(shape[0], dtype=dtype)[:, None] * jnp.ones((1, shape[1]), dtype=dtype)
indices = jnp.arange(shape[0], dtype=jnp.int32)[::-1]

# (262144, 56, 128)
x = x.reshape(shape[0], -1, LANES)
# out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)
# out_ref = x[indices, ...]

In [8]:
(4 * shape[0] * shape[1] / 1640e9) / 13e-3

0.3525418746716698

In [9]:
with jax.profiler.trace("/tmp/compute_on"):
  for _ in range(3):
    jax.block_until_ready(gather_3d_to_2d(x, indices))
    # jax.block_until_ready(combined(x, indices))

# original kernel

In [6]:
m, k = 8 * 4096, 512
keys = iter(random.split(random.key(0), 1024))
x = jnp.arange(m * k).reshape((m, k)).astype(jnp.float32)
idx = jnp.argsort(random.normal(next(keys), x.shape[0]))

In [7]:
chunk = 8  # hardware constant


@partial(jax.jit, static_argnames=("rows", "unroll_i", "unroll_j", "unroll_k"))
def gather3d_to_2d(x, idx, rows: int = 8, unroll_i: int | bool = True, unroll_j: int | bool = True, unroll_k: int | bool = False):

  if x.ndim != 3:
    x = x.reshape((x.shape[0], x.shape[1] // 128, 128))

  packing = 32 // (x.dtype.itemsize * 8)
  assert packing == 1  # for now only handle 32-bit types

  def kernel(x_ref, idx_ref, out_ref):
    tiling = ((8, 128), (1, 1))

    @partial(pl.run_scoped, scratch_ref=plsc.MemoryRef((rows, x.shape[1], x.shape[2]), x.dtype, memory_space=pltpu.VMEM, tiling=tiling))
    def _(scratch_ref):
      idx = pl.program_id(1)
      pltpu.sync_copy(x_ref.at[idx_ref, :, :], scratch_ref)

      @pl.loop(0, rows, step=packing, unroll=unroll_i)
      def _(i):
        @pl.loop(0, scratch_ref.shape[0], step=1, unroll=unroll_j)
        def _(j):
          @pl.loop(0, scratch_ref.shape[2], step=chunk, unroll=unroll_k)
          def _(k):
            j_ = scratch_ref.shape[2] * j + k
            out_ref[pl.ds(i, packing), pl.ds(j_, chunk)] = scratch_ref[pl.ds(i, packing), j, pl.ds(k, chunk)]

  out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)
  in_specs = [
    pl.BlockSpec(x.shape, lambda i, j: (0, 0, 0), memory_space=pltpu.MemorySpace.HBM),
    pl.BlockSpec((rows,), lambda i, j: i, memory_space=pltpu.MemorySpace.VMEM),
  ]
  out_specs = pl.BlockSpec((rows, x.shape[1] * x.shape[2]), lambda i, j: (i, j))

  grid = (pl.cdiv(x.shape[0], rows), pl.cdiv(x.shape[1], 8))
  print(f"{grid = }")

  pallas_gather = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
  )
  out = pallas_gather(x, idx)
  return out

In [8]:
out = gather3d_to_2d(x, idx)

grid = (4096, 1)


In [ ]:
import tune_jax
tune_jax.logger.setLevel("INFO")

hyperparams = dict(
  rows=[8, 16],
  unroll_i=[True, False],
  unroll_j=[False],
  unroll_k=False,
)

fn = tune_jax.tune(gather3d_to_2d, hyperparams=hyperparams)
_ = fn(x, idx)
print(tune_jax.tabulate(fn))

In [ ]:
with jax.profiler.trace("/tmp/gather_3d"):
  out = jax.block_until_ready(gather3d_to_2d(x, idx))

In [ ]:
print((1 * (jnp.sum(jnp.abs(out - x[idx, ...]), -1) > 0)))
print(jnp.sum((1 * (jnp.sum(jnp.abs(out - x[idx, ...]), -1) > 0))))

# Gleb's old kernel

In [2]:
LANES = 128
NUM_SUBCORES = 16
NUM_CORES = 2

shape = (262144 // 2, 8192 // 4)
dtype = jnp.float32
# x = jax.random.uniform(jax.random.key(0), shape, dtype=dtype)
x = jnp.arange(shape[0])[:, None] * jnp.ones(shape[1], dtype=dtype)[None, :]
# indices = jnp.arange(shape[0], dtype=jnp.int32)[::-1]
indices = jnp.argsort(random.uniform(jax.random.key(1), shape[0]))

# (262144, 56, 128)
x = x.reshape(shape[0], -1, LANES)
out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)

rows = 1
window = 8
chunk = 8

RuntimeError: Unable to initialize backend 'tpu': UNKNOWN: TPU initialization failed: open(/dev/vfio/0): Device or resource busy: Device or resource busy; Couldn't open iommu group /dev/vfio/0 (set JAX_PLATFORMS='' to automatically choose an available backend)

In [ ]:
@plsc.kernel(
    out_shape=out_shape,
    mesh=plsc.VectorSubcoreMesh(
          core_axis_name="core", subcore_axis_name="subcore", num_cores=2
    ),
    scratch_shapes=(
        pltpu.VMEM((window,), jnp.int32),
        pltpu.VMEM((window, x.shape[1], LANES), dtype),
        pltpu.VMEM((window, x.shape[1] * LANES), dtype),
    ),
)
def gather_3d_to_2d(x_ref, indices_ref, o_ref, indices_vmem, x_scratch_ref, o_scratch_ref):
  core_id = jax.lax.axis_index("core")
  subcore_id = jax.lax.axis_index("subcore")
  assert indices.shape[0] % (8 * NUM_SUBCORES * NUM_CORES) == 0
  subcore_slice = indices.shape[0] // (NUM_SUBCORES * NUM_CORES)

  @pl.loop(0, subcore_slice, step=window)
  def _(i):
    start_i = (core_id * NUM_SUBCORES + subcore_id) * subcore_slice + i
    start_i = pl.multiple_of(start_i, 8)
    pltpu.sync_copy(indices_ref.at[pl.ds(start_i, window)], indices_vmem)
    pltpu.sync_copy(x_ref.at[indices_vmem], x_scratch_ref)

    @pl.loop(0, x_scratch_ref.shape[0], step=rows)
    def _(j):
      @pl.loop(0, x_scratch_ref.shape[1], step=1)
      def _(k):
        @pl.loop(0, x_scratch_ref.shape[2], step=chunk)
        def _(l):
          j_ = LANES * k + l
          o_scratch_ref[pl.ds(j, rows), pl.ds(j_, chunk)] = x_scratch_ref[pl.ds(j, rows), k, pl.ds(l, chunk)]
    pltpu.sync_copy(o_scratch_ref, o_ref.at[pl.ds(start_i, 8)])

In [23]:
# import tune_jax
# fn = tune_jax.tune(gather_3d_to_2d)
# fn(x, indices)
# print(tune_jax.tabulate(fn))

In [24]:
2 * 4 * x.size / 1640e9

0.001309441248780488

In [31]:
out = gather_3d_to_2d(x, indices)
out_ref = x[indices].reshape(out_shape.shape)

In [32]:
jnp.sum(jnp.abs(out - out_ref), -1)

Array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [33]:
out

Array([[ 26622.,  26622.,  26622., ...,  26622.,  26622.,  26622.],
       [  9729.,   9729.,   9729., ...,   9729.,   9729.,   9729.],
       [ 14066.,  14066.,  14066., ...,  14066.,  14066.,  14066.],
       ...,
       [   582.,    582.,    582., ...,    582.,    582.,    582.],
       [100276., 100276., 100276., ..., 100276., 100276., 100276.],
       [ 95444.,  95444.,  95444., ...,  95444.,  95444.,  95444.]],      dtype=float32)

In [34]:
xla_gather = jax.jit(lambda x, idx: x[idx])
_ = xla_gather(x, indices)
with jax.profiler.trace("/tmp/gather"):
  for _ in range(3):
    out = jax.block_until_ready(gather_3d_to_2d(x, indices))
  for _ in range(3):
    _ = jax.block_until_ready(xla_gather(x, indices))

In [35]:
out_ref

Array([[ 26622.,  26622.,  26622., ...,  26622.,  26622.,  26622.],
       [  9729.,   9729.,   9729., ...,   9729.,   9729.,   9729.],
       [ 14066.,  14066.,  14066., ...,  14066.,  14066.,  14066.],
       ...,
       [   582.,    582.,    582., ...,    582.,    582.,    582.],
       [100276., 100276., 100276., ..., 100276., 100276., 100276.],
       [ 95444.,  95444.,  95444., ...,  95444.,  95444.,  95444.]],      dtype=float32)

In [36]:
idxs_matches = jnp.where(jnp.sum(jnp.abs(out - out_ref) > 1e-3, axis=-1) == 0)
print(f"rows matches: {len(idxs_matches[0]) / x.shape[0]:.2%}")

rows matches: 100.00%


In [ ]:
LANES = 128
NUM_SUBCORES = 16
NUM_CORES = 2

# pad to multiples of (8 * NUM_SUBCORES)
# shape = (262144, 7168 // 2)
# shape = (262144 // 8, 7168 // 2)
shape = (262144 // 8, 8192 // 2)
# dtype = jnp.bfloat16
dtype = jnp.float32
# x = jax.random.uniform(jax.random.key(0), shape, dtype=dtype)
x = jnp.arange(shape[0], dtype=dtype)[:, None] * jnp.ones((1, shape[1]), dtype=dtype)
# indices = jnp.arange(shape[0], dtype=jnp.int32)[::-1]
indices = jnp.arange(shape[0], dtype=jnp.int32)

# (262144, 56, 128)
x = x.reshape(shape[0], -1, LANES)
out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)
out_ref = x[indices, ...]

# tiling = ((8, 128), (1, 1))
# scratch_ref = lambda shape: plsc.MemoryRef(shape, x.dtype, memory_space=pltpu.VMEM, tiling=tiling)

# Restarted attempt

In [1]:
def kernel_with_scratch(x_ref, idx_ref, out_ref, x_scratch_ref):
  rows = 1
  chunk = 8
  cols = 8
  pltpu.sync_copy(x_ref.at[idx_ref[...]], x_scratch_ref)

  @pl.loop(0, x_scratch_ref.shape[0], step=rows)
  def _(j):
    @pl.loop(0, x_scratch_ref.shape[1], step=1)
    def _(k):
      @pl.loop(0, x_scratch_ref.shape[2], step=chunk)
      def _(l):
        j_ = k * x_scratch_ref.shape[2] + l
        # out_ref[pl.ds(j, rows), pl.ds(j_, chunk)] = x_scratch_ref[pl.ds(j, rows), pl.ds(k, 1), pl.ds(l, chunk)].reshape((rows, -1))
        # out_ref[rows, pl.ds(j_, cols * chunk)] = x_scratch_ref[rows, k, pl.ds(l, cols * chunk)].reshape((-1,))
        out_ref[rows, pl.ds(j_, cols * chunk)] = x_scratch_ref[rows, k, pl.ds(l, cols * chunk)].reshape((-1,))


def kernel(x_ref, idx_ref, out_ref):
  tiling = ((8, 128), (1, 1))
  pl.run_scoped(
    partial(kernel_with_scratch, x_ref, idx_ref, out_ref),
    x_scratch_ref=plsc.MemoryRef(
      (idx_ref.shape[0], *x.shape[1:]), x_ref.dtype, memory_space=pltpu.VMEM, tiling=tiling
    )
  )


@jax.jit
def gather_3d_to_2d(x, idx):
  rows = 8
  out_shape = jax.ShapeDtypeStruct((idx.shape[0], math.prod(x.shape[1:])), x.dtype)
  in_specs = [pl.BlockSpec(memory_space=pltpu.HBM), pl.BlockSpec((rows,), lambda i: (i,))]
  out_specs = pl.BlockSpec((rows, out_shape.shape[1]), lambda i: (i, 0))
  grid = (pl.cdiv(x.shape[0], rows),)
  return pl.pallas_call(
    kernel, out_shape=out_shape, in_specs=in_specs, out_specs=out_specs, grid=grid,
    compiler_params=pltpu.CompilerParams(
      dimension_semantics=["arbitrary"], kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE
    )
  )(x, idx)

NameError: name 'jax' is not defined

In [ ]:
LANES = 128
shape = (262144 // 8, 8192 // 4)
dtype = jnp.float32
x = jnp.arange(shape[0], dtype=dtype)[:, None] * jnp.ones((1, shape[1]), dtype=dtype)
# indices = jnp.arange(shape[0], dtype=jnp.int32)[::-1]
indices = jnp.argsort(jax.random.uniform(jax.random.key(1), shape[:1]))

x = x.reshape(shape[0], -1, LANES)
out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)
out_ref = x[indices, ...].reshape((indices.shape[0], -1))

In [ ]:
y = gather_3d_to_2d(x, indices)
gather_xla = compute_on("tpu_sparsecore")(jax.jit(lambda x, idx: x[idx, ...].reshape((idx.shape[0], -1))))
_ = gather_xla(x, indices)
with jax.profiler.trace("/tmp/gather"):
  for _ in range(3):
    _ = jax.block_until_ready(gather_3d_to_2d(x, indices))
  for _ in range(3):
    _ = jax.block_until_ready(gather_xla(x, indices))

In [ ]:
jnp.sum(y[0:, :] - out_ref[:, :])

Array(4.149757e+10, dtype=float32)

In [ ]:
y[:8, :5]

Array([[31315., 31315., 31315., 31315., 31315.],
       [ 9729.,  9729.,  9729.,  9729.,  9729.],
       [ 9729.,  9729.,  9729.,  9729.,  9729.],
       [31315., 31315., 31315., 31315., 31315.],
       [31315., 31315., 31315., 31315., 31315.],
       [31315., 31315., 31315., 31315., 31315.],
       [31315., 31315., 31315., 31315., 31315.],
       [31315., 31315., 31315., 31315., 31315.]], dtype=float32)

In [ ]:
out_ref[:8, :5]

Array([[26622., 26622., 26622., 26622., 26622.],
       [ 9729.,  9729.,  9729.,  9729.,  9729.],
       [14066., 14066., 14066., 14066., 14066.],
       [12993., 12993., 12993., 12993., 12993.],
       [16658., 16658., 16658., 16658., 16658.],
       [29271., 29271., 29271., 29271., 29271.],
       [15034., 15034., 15034., 15034., 15034.],
       [ 8559.,  8559.,  8559.,  8559.,  8559.]], dtype=float32)

In [23]:
jnp.max(jnp.abs(out_ref - y), axis=-1)

Array([3.2767e+04, 3.2766e+04, 3.2765e+04, ..., 2.0000e+00, 1.0000e+00,
       0.0000e+00], dtype=float32)

In [24]:
y[-10:, :5]

Array([[32756., 32756., 32756., 32756., 32756.],
       [32756., 32756., 32756., 32756., 32756.],
       [32760., 32760., 32760., 32760., 32760.],
       [32760., 32760., 32760., 32760., 32760.],
       [32760., 32760., 32760., 32760., 32760.],
       [32760., 32760., 32760., 32760., 32760.],
       [32764., 32764., 32764., 32764., 32764.],
       [32764., 32764., 32764., 32764., 32764.],
       [32764., 32764., 32764., 32764., 32764.],
       [32764., 32764., 32764., 32764., 32764.]], dtype=float32)

In [25]:
rows = 4
window = 8
chunk = 16

# Gleb's kernel

In [ ]:

LANES = 128
NUM_SUBCORES = 16
NUM_CORES = 2

shape = (262144 // 8, 8192 // 2)
dtype = jnp.float32
x = jnp.arange(shape[0], dtype=dtype)[:, None] * jnp.ones((1, shape[1]), dtype=dtype)
indices = jnp.arange(shape[0], dtype=jnp.int32)

x = x.reshape(shape[0], -1, LANES)
out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)
out_ref = x[indices, ...]

rows = 4
window = 8
chunk = 16


@jax.jit
@plsc.kernel(
    out_shape=jax.ShapeDtypeStruct((x.shape[0], 8, x.shape[2]), x.dtype),
    mesh=plsc.VectorSubcoreMesh(
          core_axis_name="core", subcore_axis_name="subcore", num_cores=1
    ),
    scratch_shapes=(
        pltpu.VMEM((window,), jnp.int32),
        pltpu.VMEM((window, 8, LANES), dtype),
        # pltpu.VMEM((window, x.shape[1] * LANES), dtype),
        pltpu.VMEM((window, 8, LANES), dtype),
    ),
)
def gather_3d(x_ref, indices_ref, o_ref, indices_vmem, x_scratch_ref, o_scratch_ref):
  """Attempt to copy just the first 8 sublanes into the output with a gathered first dimension."""
  core_id = jax.lax.axis_index("core")
  subcore_id = jax.lax.axis_index("subcore")
  assert indices.shape[0] % (8 * NUM_SUBCORES * NUM_CORES) == 0
  subcore_slice = indices.shape[0] // (NUM_SUBCORES * NUM_CORES)

  @pl.when(subcore_id == 0)
  def _():
    start_i = (core_id * NUM_SUBCORES + subcore_id) * subcore_slice + 0
    start_i = pl.multiple_of(start_i, 8)
    pltpu.sync_copy(indices_ref.at[pl.ds(start_i, window)], indices_vmem)

    # pltpu.sync_copy(x_ref.at[indices_vmem].at[:, pl.ds(0, x_scratch_ref.shape[1]), :], x_scratch_ref)
    pltpu.sync_copy(x_ref.at[:, pl.ds(0, x_scratch_ref.shape[1]), :].at[indices_vmem], x_scratch_ref)
    pltpu.sync_copy(x_scratch_ref, o_ref.at[pl.ds(start_i, window)])
    return


out = gather_3d(x, indices)
jnp.sum(out)

AttributeError: module 'jax.experimental.pallas.tpu_sc' has no attribute 'kernel'

In [15]:
x.shape

(32768, 32, 128)

In [11]:
# @partial(jax.jit, compiler_options=dict(xla_tpu_enable_sc_log_recorder="true", xla_tpu_enable_tile_log_recorder="true"))
@jax.jit
@plsc.kernel(
    # out_shape=out_shape,
    out_shape=jax.ShapeDtypeStruct((x.shape[0], 8, x.shape[2]), x.dtype),
    mesh=plsc.VectorSubcoreMesh(
          core_axis_name="core", subcore_axis_name="subcore", num_cores=1
    ),
    scratch_shapes=(
        pltpu.VMEM((window,), jnp.int32),
        pltpu.VMEM((window, 8, LANES), dtype),
        # pltpu.VMEM((window, x.shape[1] * LANES), dtype),
        pltpu.VMEM((window, 8, LANES), dtype),
    ),
)
def gather_3d_to_2d(x_ref, indices_ref, o_ref, indices_vmem, x_scratch_ref, o_scratch_ref):
  core_id = jax.lax.axis_index("core")
  subcore_id = jax.lax.axis_index("subcore")
  assert indices.shape[0] % (8 * NUM_SUBCORES * NUM_CORES) == 0
  subcore_slice = indices.shape[0] // (NUM_SUBCORES * NUM_CORES)

  start_i = (core_id * NUM_SUBCORES + subcore_id) * subcore_slice + 0
  start_i = pl.multiple_of(start_i, 8)
  pltpu.sync_copy(indices_ref.at[pl.ds(start_i, window)], indices_vmem)

  # pltpu.sync_copy(x_ref.at[indices_vmem].at[:, pl.ds(0, x_scratch_ref.shape[1]), :], x_scratch_ref)
  pltpu.sync_copy(x_ref.at[:, pl.ds(0, x_scratch_ref.shape[1]), :].at[indices_vmem], x_scratch_ref)
  # pltpu.sync_copy(x_scratch_ref, o_ref.at[pl.ds(start_i, window)].at[:, pl.ds(0, x_scratch_ref.shape[1]), :])
  pltpu.sync_copy(x_scratch_ref, o_ref.at[pl.ds(start_i, window)])
  return

  # @pl.when((subcore_id == 0))
  # def _():
  #   @pl.loop(0, subcore_slice, step=window)
  #   def _(i):
  #     start_i = (core_id * NUM_SUBCORES + subcore_id) * subcore_slice  + i
  #     # start_i = pl.multiple_of(start_i, 8)
  #     pltpu.sync_copy(indices_ref.at[pl.ds(start_i, window)], indices_vmem)
  #     pltpu.sync_copy(x_ref.at[indices_vmem], o_ref.at[pl.ds(start_i, window)])
  #     return

  #     start_i = 0
  #     pltpu.sync_copy(indices_ref.at[pl.ds(start_i, window)], indices_vmem)
  #     pltpu.sync_copy(x_ref.at[indices_vmem], x_scratch_ref)
  #     # @pl.loop(0, x_scratch_ref.shape[0], step=rows)
  #     # def _(j):
  #     #   @pl.loop(0, x_scratch_ref.shape[2], step=chunk)
  #     #   def _(l):
  #     #     @pl.loop(0, x_scratch_ref.shape[1], step=1)
  #     #     def _(k):
  #     for j in range(0, x_scratch_ref.shape[0], rows):
  #       for l in range(0, x_scratch_ref.shape[2], chunk):
  #         for k in range(0, x_scratch_ref.shape[1]):
  #           j_ = LANES * k + l
  #           o_scratch_ref[pl.ds(j, rows), pl.ds(j_, chunk)] = x_scratch_ref[pl.ds(j, rows), k, pl.ds(l, chunk)]
  #     #pl.debug_print("scratch_ref", o_scratch_ref[0, 0])
  #     pltpu.sync_copy(o_scratch_ref, o_ref.at[pl.ds(start_i, 8)])

In [12]:
out = gather_3d_to_2d(x, indices)

In [13]:
out[:10, :3]

JaxRuntimeError: INTERNAL: Program or fatal error occurred; computation may be invalid: Assertion args: INTERNAL: Accelerator device halted prematurely, perhaps due to an on-device check-failure. Node 0 halted unexpectedly at tag:pc SparseCoreSequencer:0:0xfb (from SparseCoreSequencer:0:0xfb): no debugging message found for this tag:pc. 
  0x0x0_SC0 SparseCoreTileExecuteCoreSequencer:11 halted at tag:pc SparseCoreTileExecuteCoreSequencer:1:0xa1 (from SparseCoreTileExecuteCoreSequencer:1:0xa1): no debugging message found for this tag:pc. 
  0x0x0_SC0 SparseCoreTileExecuteCoreSequencer:10 halted at tag:pc SparseCoreTileExecuteCoreSequencer:1:0xa1 (from SparseCoreTileExecuteCoreSequencer:1:0xa1): no debugging message found for this tag:pc. 
  0x0x0_SC0 SparseCoreTileExecuteCoreSequencer:4 halted at tag:pc SparseCoreTileExecuteCoreSequencer:1:0xa1 (from SparseCoreTileExecuteCoreSequencer:1:0xa1): no debugging message found for this tag:pc. 
  0x0x0_SC0 SparseCoreTileExecuteCoreSequencer:7 

In [12]:
x[:, :3]

Array([[[0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00,
         0.0000e+00, 0.0000e+00]],

       [[1.0000e+00, 1.0000e+00, 1.0000e+00, ..., 1.0000e+00,
         1.0000e+00, 1.0000e+00],
        [1.0000e+00, 1.0000e+00, 1.0000e+00, ..., 1.0000e+00,
         1.0000e+00, 1.0000e+00],
        [1.0000e+00, 1.0000e+00, 1.0000e+00, ..., 1.0000e+00,
         1.0000e+00, 1.0000e+00]],

       [[2.0000e+00, 2.0000e+00, 2.0000e+00, ..., 2.0000e+00,
         2.0000e+00, 2.0000e+00],
        [2.0000e+00, 2.0000e+00, 2.0000e+00, ..., 2.0000e+00,
         2.0000e+00, 2.0000e+00],
        [2.0000e+00, 2.0000e+00, 2.0000e+00, ..., 2.0000e+00,
         2.0000e+00, 2.0000e+00]],

       ...,

       [[3.2765e+04, 3.2765e+04, 3.2765e+04, ..., 3.2765e+04,
         3.2765e+04, 3.2765e+04],
        [3.2765e+04, 

In [ ]:
def gather_3d_to_2d_v2(x, indices):
  def kernel(x_ref, indices_ref, o_ref, indices_vmem, x_scratch_ref, o_scratch_ref):
    core_id = 0
    subcore_id = 0
    assert indices.shape[0] % (8 * NUM_SUBCORES * NUM_CORES) == 0
    subcore_slice = indices.shape[0] // (NUM_SUBCORES * NUM_CORES)
    pl.debug_print("", o_scratch_ref)
    # @pl.loop(0, subcore_slice, step=window)
    # def _(i):
    for i in range(1):
      start_i = (core_id * NUM_SUBCORES + subcore_id) * subcore_slice + i
      start_i = pl.multiple_of(start_i, 8)
      # pltpu.sync_copy(indices_ref.at[pl.ds(start_i, window)], indices_vmem)
      # pltpu.sync_copy(x_ref.at[indices_vmem], x_scratch_ref)
      # pltpu.sync_copy(x_ref.at[indices[pl.ds(start_i, window)]], x_scratch_ref)

      pltpu.sync_copy(x_ref.at[indices_ref], x_scratch_ref)
      # @pl.loop(0, x_scratch_ref.shape[0], step=rows)
      # def _(j):
      #  @pl.loop(0, x_scratch_ref.shape[2], step=chunk)
      #  def _(l):
      #    @pl.loop(0, x_scratch_ref.shape[1], step=1)
      #    def _(k):
      for j in range(0, x_scratch_ref.shape[0], rows):
        for l in range(0, x_scratch_ref.shape[2], chunk):
          for k in range(0, x_scratch_ref.shape[1]):
            j_ = LANES * k + l
            o_scratch_ref[pl.ds(j, rows), pl.ds(j_, chunk)] = x_scratch_ref[pl.ds(j, rows), k, pl.ds(l, chunk)]
      pltpu.sync_copy(o_scratch_ref, o_ref.at[pl.ds(start_i, 8)])

  return pl.pallas_call(
    kernel,
    grid=(1,),
    out_shape=jax.ShapeDtypeStruct((x.shape[0], x.shape[1] * x.shape[2]), x.dtype),
    in_specs=[pl.BlockSpec(memory_space=pltpu.HBM), pl.BlockSpec((window,), lambda i: (i,), memory_space=pltpu.VMEM)],
    out_specs=pl.BlockSpec(memory_space=pltpu.HBM),
    scratch_shapes=[
      pltpu.VMEM((window,), jnp.int32),
      pltpu.VMEM((window, x.shape[1], LANES), dtype),
      pltpu.VMEM((window, x.shape[1] * LANES), dtype),
    ],
    compiler_params=pltpu.CompilerParams(kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
  )(x, indices)

In [25]:
fn = jax.jit(gather_3d_to_2d_v2, compiler_options=dict(xla_tpu_enable_sc_log_recorder="true", xla_tpu_enable_tile_log_recorder="true"))
out2 = fn(x, indices)

TypeError: CompilerParams.__init__() got an unexpected keyword argument 'num_cores'

In [26]:
out2[:window, :]

Array([[0., 0., 0., ..., 4., 4., 4.],
       [0., 0., 0., ..., 4., 4., 4.],
       [0., 0., 0., ..., 4., 4., 4.],
       ...,
       [0., 0., 0., ..., 4., 4., 4.],
       [0., 0., 0., ..., 4., 4., 4.],
       [0., 0., 0., ..., 4., 4., 4.]], dtype=float32)

In [20]:
jnp.unique(out2[:window, :])

Array([0., 3., 4.], dtype=float32)

In [17]:
np.unique(np.array(out2[:window, :]))

array([0., 3., 4.], dtype=float32)

In [19]:
np.unique(np.array(x[:window, :]))

array([0., 1., 2., 3., 4., 5., 6., 7.], dtype=float32)

In [ ]:
x[0, :]

Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [32]:
out2[0, :].reshape((-1, 128))

Array([[ 7.,  7.,  7., ...,  7.,  7.,  7.],
       [ 7.,  7.,  7., ...,  7.,  7.,  7.],
       [ 7.,  7.,  7., ...,  7.,  7.,  7.],
       ...,
       [10., 10., 10., ..., 10., 10., 10.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [11., 11., 11., ..., 11., 11., 11.]], dtype=float32)

In [33]:
jnp.all(out2[0, :] == 0)

Array(False, dtype=bool)

In [34]:
with jnp.printoptions(threshold=1e9):
  print(out2[0, :])

[ 7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.
  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7.  7

In [107]:
out[0, :].reshape((-1, 128))[:, 2]

Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 3., 4.,
       4., 3., 0., 4., 3., 3., 4., 4., 3., 0., 4.], dtype=float32)

In [110]:
out[:-3, :]

Array([[  0.,   0.,   0., ...,   4.,   4.,   4.],
       [  0.,   0.,   0., ...,   4.,   4.,   4.],
       [  0.,   0.,   0., ...,   4.,   4.,   4.],
       ...,
       [-29., -29., -29., ...,  -2.,  -2.,  -2.],
       [-28., -28., -28., ...,  -1.,  -1.,  -1.],
       [-27., -27., -27., ...,   0.,   0.,   0.]], dtype=float32)

In [75]:
idx1, idx2 = jnp.where(jnp.linalg.norm(out.reshape((out.shape[0], -1, 128)) - out_ref, axis=-1) == 0)
len(idx1) / x.shape[0]

0.003997802734375

In [29]:
out[0, :].reshape((-1, 128))[0, :32]

Array([0.7202661 , 0.0688026 , 0.3571427 , 0.2957555 , 0.24287021,
       0.79572034, 0.9621483 , 0.8600738 , 0.02221012, 0.6904905 ,
       0.3961414 , 0.31373   , 0.17692256, 0.9225285 , 0.43593323,
       0.95272076, 0.60985196, 0.55133367, 0.9549476 , 0.74515975,
       0.56910396, 0.18761683, 0.62160313, 0.9821942 , 0.5527153 ,
       0.9309175 , 0.9383259 , 0.523155  , 0.6520113 , 0.71207035,
       0.9351195 , 0.95415354], dtype=float32)

In [30]:
out

Array([[0.7202661 , 0.0688026 , 0.3571427 , ..., 0.31876385, 0.92386913,
        0.5301248 ],
       [0.50162745, 0.24450302, 0.47794557, ..., 0.82582223, 0.5364537 ,
        0.01031697],
       [0.23478484, 0.5940739 , 0.7713307 , ..., 0.12427711, 0.43438208,
        0.664691  ],
       ...,
       [0.5707718 , 0.42756116, 0.9239912 , ..., 0.77879965, 0.04856896,
        0.3872732 ],
       [0.98550236, 0.27016842, 0.28292155, ..., 0.86440897, 0.50670874,
        0.5434263 ],
       [0.70419025, 0.19259083, 0.42129123, ..., 0.5049288 , 0.52444136,
        0.08704817]], dtype=float32)

In [33]:
out - out_ref.reshape((out.shape[0], -1))

Array([[ 0.69570374, -0.6255666 , -0.2872957 , ..., -0.00831318,
         0.21955395,  0.0381403 ],
       [-0.1395706 ,  0.01888001, -0.19609797, ...,  0.3072251 ,
        -0.32353032, -0.70233047],
       [-0.500911  ,  0.04123044,  0.66437995, ..., -0.51618266,
        -0.4389863 , -0.02710617],
       ...,
       [ 0.40185332, -0.55527437,  0.20238781, ...,  0.28696406,
        -0.47866392, -0.43795323],
       [ 0.61376953, -0.53452015,  0.20999849, ..., -0.06981766,
         0.17103481, -0.32232237],
       [-0.24347675, -0.78598905,  0.08899975, ...,  0.        ,
         0.        ,  0.        ]], dtype=float32)

In [56]:
diff = out_ref.reshape((out_ref.shape[0], -1))[:8, :] - out[:8, :]

In [57]:
jnp.linalg.norm(diff, axis=-1)

Array([26.526152, 27.060911, 26.181795, 27.43073 , 26.25917 , 25.427439,
       26.513681, 25.235216], dtype=float32)

In [13]:
idx1[:10]

Array([ 0,  7,  8, 15, 16, 23, 24, 31, 32, 39], dtype=int32)

In [40]:
idx1[-1]

Array(261119, dtype=int32)

In [40]:
out.shape[0] / 97920

2.6666666666666665

In [9]:
out_ref[0, :][0:16, 0]

Array([0.117188, 0.0234375, 0.546875, 0.0390625, 0.570312, 0.398438,
       0.757812, 0.492188, 0.65625, 0.578125, 0.9375, 0.429688, 0.421875,
       0.625, 0.679688, 0.875], dtype=bfloat16)

In [10]:
out_ref[0, :][0:16, 0] - out[0, :].reshape((-1, 128))[0:16, 0]

Array([-0.765625, -0.5, 0.0078125, -0.390625, -0.078125, -0.0234375,
       0.71875, 0.015625, -0.132812, -0.226562, 0.820312, -0.398438, -0.5,
       0.507812, -0.195312, -0.0703125], dtype=bfloat16)

In [5]:
jnp.linalg.norm(x[indices, ...] - out.reshape((out.shape[0], -1, 128)), axis=-1)

Array([[0, 4.5, 5.0625, ..., 6.21875, 6.25, 6.5625],
       [4.84375, 4.96875, 4.5625, ..., 6.3125, 6.65625, 6.75],
       [4.75, 4.34375, 4.5625, ..., 6.03125, 6.25, 6.25],
       ...,
       [4.46875, 4.59375, 4.6875, ..., 7, 6.90625, 6.15625],
       [5.03125, 4.90625, 5.21875, ..., 6.46875, 6.71875, 6.40625],
       [4.46875, 4.15625, 5.03125, ..., 6.90625, 6.71875, 6.65625]],      dtype=bfloat16)

In [6]:
out[0, :].reshape((-1, 128))[4, :32]

Array([0.101562, 0.875, 0.382812, 0.960938, 0.0390625, 0.929688, 0.601562,
       0.367188, 0.140625, 0.601562, 0.914062, 0.1875, 0.390625, 0.390625,
       0.585938, 0.304688, 0.851562, 0.554688, 0.914062, 0.8125, 0.617188,
       0.320312, 0.0390625, 0.101562, 0.734375, 0.234375, 0.890625,
       0.203125, 0.148438, 0.289062, 0.101562, 0.09375], dtype=bfloat16)

In [7]:
x[indices, ...][0, :, :][4, :32]

Array([0.570312, 0.484375, 0.570312, 0.90625, 0.960938, 0.851562,
       0.601562, 0.398438, 0.234375, 0.640625, 0.726562, 0.148438, 0.125,
       0.820312, 0.484375, 0.0078125, 0.570312, 0.09375, 0.851562,
       0.421875, 0.078125, 0.5625, 0.671875, 0.09375, 0.414062, 0.945312,
       0.726562, 0.03125, 0.25, 0.109375, 0.148438, 0.140625],      dtype=bfloat16)

In [8]:
np.testing.assert_array_equal(kernel(x, indices), x[indices].reshape((x.shape[0], -1)))

AssertionError: 
Arrays are not equal

Mismatched elements: 1852938902 / 1871708160 (99%)
Max absolute difference among violations: 0.992188
Max relative difference among violations: 126
 ACTUAL: array([[0.117188, 0.507812, 0.648438, ..., 0, 0, 0],
       [0.0234375, 0.125, 0.101562, ..., 0, 0, 0],
       [0.546875, 0.59375, 0, ..., 0, 0, 0],...
 DESIRED: array([[0.117188, 0.507812, 0.648438, ..., 0.867188, 0.921875, 0.320312],
       [0.882812, 0.421875, 0.84375, ..., 0.125, 0.148438, 0.492188],
       [0.226562, 0.28125, 0.648438, ..., 0.671875, 0.414062, 0.140625],...

In [28]:
with jax.profiler.trace("/tmp/gather_gleb"):
  for _ in range(5):
    jax.block_until_ready(kernel(x, indices))
    jax.block_until_ready(x[indices])